# 마스킹된 부분 표면 이미지 GAN 생성 예비 시험

기존 마스킹 전처리로 베드를 가리고 남긴 실제 언더필 이미지 50장으로 DCGAN을 학습해 이미지를 50장 생성합니다. 이번에는 **분류기 학습을 하지 않습니다.** 학습 이미지는 원본 1000×1000 크롭을 기존 방식으로 마스킹한 뒤 256×256으로 줄였습니다. 축소에는 LANCZOS 보간을 사용했으며, 이전 분류기 코드의 bilinear 보간과는 다릅니다. 이미지의 회색 영역도 생성기 입력에 포함됩니다.

직전 베드 포함 실험과 같은 원본은 46장이고 나머지 4장은 기존 유효 마스크가 있는 다른 학습 사진입니다. 따라서 두 실행의 차이를 마스킹 효과만으로 해석할 수 없습니다. 데이터 출처와 선택 조건은 데이터 ZIP의 manifest를 확인하세요. 라벨은 출력 조건이며 모든 사진에 눈에 보이는 결함이 있다는 보장은 없습니다.

Colab 런타임 유형에서 G4를 선택한 뒤 아래 셀을 실행하세요. 1,500단계는 이전과 같은 임시 설정이며 품질을 보장하지 않습니다. 입력·생성 사진과 학습 로그 및 결과 ZIP을 Drive에 저장하고, 정상 종료 및 가능한 오류 상황에서 Drive를 동기화한 뒤 런타임을 자동 반납합니다. 중간에 강제 종료되거나 연결 장애가 생기면 자동 반납이 보장되지 않으므로 상태를 확인해야 합니다.

In [ ]:
from google.colab import runtime, drive
import traceback, sys, os, json, hashlib, time, subprocess, zipfile
from pathlib import Path, PurePosixPath

MOUNTED = False
RUN_ROOT = None
try:
    drive.mount('/content/drive')
    MOUNTED = True
    import numpy as np
    import torch
    from PIL import Image, ImageOps, ImageDraw
    from IPython.display import display

    DRIVE = Path('/content/drive/MyDrive/산공종설')
    DATA_ZIP = DRIVE / 'gan_masked_50_data_20261101.zip'
    CODE_ZIP = DRIVE / 'gan_masked_pilot_code_20261101.zip'
    RUN_ROOT = DRIVE / 'results/gan_masked_20261101_run01'
    PREPARED, GAN = RUN_ROOT/'prepared', RUN_ROOT/'gan'
    PREVIEW, LOGS = RUN_ROOT/'review', RUN_ROOT/'logs'
    CODE_DIR = Path('/content/gan_masked_code')
    SEED, STEPS, BATCH = 42, 1500, 8
    for item in (DATA_ZIP, CODE_ZIP):
        if not item.is_file(): raise FileNotFoundError(f'필요한 Drive 파일이 없습니다: {item}')
    if not torch.cuda.is_available(): raise RuntimeError('GPU가 없습니다. G4를 선택하세요.')
    gpu = torch.cuda.get_device_name(0)
    print('사용 GPU:', gpu, flush=True)
    if 'RTX PRO 6000' not in gpu.upper():
        raise RuntimeError(f'G4 GPU가 아닙니다: {gpu}. G4를 선택하세요.')

    def sha(path):
        h = hashlib.sha256()
        with Path(path).open('rb') as stream:
            for block in iter(lambda: stream.read(4*1024*1024), b''):
                h.update(block)
        return h.hexdigest()
    if sha(CODE_ZIP) != '55748570a236233a25d1f459453edebf1c7bf2311ecd37c002eac55fb3a81b32' or sha(DATA_ZIP) != '885634654b0e727ded77beaa505fe7c9089eb22828ecb481b51ba6efeebe4dc5':
        raise ValueError('노트북에 기록된 코드 또는 데이터 묶음과 Drive 파일이 다릅니다.')
    settings = {'seed':SEED,'steps':STEPS,'batch':BATCH,'gpu':gpu,
                'dataset_sha256':sha(DATA_ZIP),'code_sha256':sha(CODE_ZIP),
                'scope':'masked surface generation only; no classifier', 'image_count':50}
    RUN_ROOT.mkdir(parents=True,exist_ok=True)
    PREVIEW.mkdir(exist_ok=True); LOGS.mkdir(exist_ok=True)
    lock = RUN_ROOT/'run_settings.json'
    if lock.exists():
        old=json.loads(lock.read_text(encoding='utf-8'))
        for key in ('seed','steps','batch','dataset_sha256','code_sha256'):
            if old[key] != settings[key]: raise ValueError('기존 실험과 설정이 다릅니다. 새 RUN_NAME이 필요합니다.')
    else:
        lock.write_text(json.dumps(settings,ensure_ascii=False,indent=2),encoding='utf-8')

    CODE_DIR.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(CODE_ZIP) as z:
        if set(z.namelist()) != {'gan_model.py'}: raise ValueError('코드 ZIP 구성이 예상과 다릅니다.')
        (CODE_DIR/'gan_model.py').write_bytes(z.read('gan_model.py'))

    # 직접 검증한 데이터 묶음만 안전한 상대 경로로 풀고 기존 결과는 덮어쓰지 않는다.
    marker=PREPARED/'.data_complete.json'
    if marker.exists():
        if json.loads(marker.read_text(encoding='utf-8')).get('dataset_sha256') != settings['dataset_sha256']:
            raise RuntimeError('기존 데이터의 출처가 달라 실행을 중단합니다.')
        print('기존 데이터 준비 결과 재사용',flush=True)
    else:
        if PREPARED.exists() and any(PREPARED.iterdir()):
            raise RuntimeError('데이터 준비가 중단된 기존 폴더입니다. 기존 결과를 보존하고 새 이름으로 실행하세요.')
        PREPARED.mkdir(parents=True,exist_ok=True)
        with zipfile.ZipFile(DATA_ZIP) as z:
            for entry in z.infolist():
                parts=PurePosixPath(entry.filename).parts
                if not parts or entry.filename.startswith('/') or any(x in ('..','') for x in parts) or '\\' in entry.filename:
                    raise ValueError(f'안전하지 않은 데이터 ZIP 경로: {entry.filename}')
                if entry.is_dir():
                    (PREPARED.joinpath(*parts)).mkdir(parents=True,exist_ok=True)
                    continue
                dest=PREPARED.joinpath(*parts)
                dest.parent.mkdir(parents=True,exist_ok=True)
                dest.write_bytes(z.read(entry))
        marker.write_text(json.dumps({'dataset_sha256':settings['dataset_sha256']}),encoding='utf-8')
    data=PREPARED/'data/train_real/underfill'
    actual=sorted(data.glob('*.png'))
    if len(actual)!=50: raise RuntimeError(f'실사 입력 {len(actual)}장; 정확히 50장이 필요합니다: {data}')
    print('마스킹된 실제 사진 50장 준비 완료',flush=True)

    def run_logged(command, name):
        print('실행:', ' '.join(str(x) for x in command),flush=True)
        with (LOGS/name).open('a',encoding='utf-8') as log:
            log.write('\n--- '+time.strftime('%Y-%m-%d %H:%M:%S')+' ---\n'); log.flush()
            proc=subprocess.Popen([str(x) for x in command],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                                  text=True,encoding='utf-8',errors='replace',bufsize=1)
            try:
                for line in proc.stdout:
                    log.write(line); log.flush(); print(line,end='',flush=True)
                result=proc.wait()
            except BaseException:
                proc.terminate(); raise
            if result: raise RuntimeError(f'GAN 실행 실패({result}). 로그: {name}')

    run_logged([sys.executable,CODE_DIR/'gan_model.py','--data',data,'--out',GAN,
                '--steps',STEPS,'--seed',SEED,'--num-generate',50,'--device','cuda',
                '--batch',BATCH,'--save-every',100,'--log-every',25],'gan.log')
    progress=json.loads((GAN/'progress.json').read_text(encoding='utf-8'))
    manifest=json.loads((GAN/'training_manifest.json').read_text(encoding='utf-8'))
    generated=sorted((GAN/'generated').glob('*.png'))
    if progress.get('status')!='completed' or progress.get('step')!=STEPS or len(generated)!=50:
        raise RuntimeError('GAN이 완료되지 않았거나 생성 이미지가 50장이 아닙니다.')
    if {r['filename']:r['sha256'] for r in manifest['images']}!={p.name:sha(p) for p in actual}:
        raise RuntimeError('GAN 학습 목록과 실제 사진 목록의 지문이 다릅니다.')

    def make_sheet(paths,dest):
        side=180; cap=24; cols=10; rows=(len(paths)+cols-1)//cols
        sheet=Image.new('RGB',(side*cols,(side+cap)*rows),'white'); draw=ImageDraw.Draw(sheet)
        for i,p in enumerate(paths):
            x=(i%cols)*side; y=(i//cols)*(side+cap)
            with Image.open(p) as img:
                tile=ImageOps.fit(img.convert('RGB'),(side,side),method=Image.Resampling.LANCZOS)
            sheet.paste(tile,(x,y)); draw.text((x+3,y+side+3),p.stem[:24],fill='black')
        sheet.save(dest,quality=92)
    make_sheet(actual,PREVIEW/'real_masked_50.jpg')
    make_sheet(generated,PREVIEW/'generated_masked_50.jpg')
    def thumb(p):
        with Image.open(p) as img: return np.asarray(img.convert('RGB').resize((64,64)),dtype=np.float32)/255
    real_arr=np.stack([thumb(p) for p in actual]); pairs=[]
    for p in generated:
        mse=((real_arr-thumb(p))**2).mean(axis=(1,2,3)); idx=int(mse.argmin())
        pairs.append((p,actual[idx],float(mse[idx])))
    import csv
    with (PREVIEW/'nearest_training_pairs.csv').open('w',encoding='utf-8-sig',newline='') as f:
        w=csv.writer(f); w.writerow(['generated','nearest_real_training','pixel_mse_64'])
        w.writerows([(a.name,b.name,d) for a,b,d in pairs])
    for k in range(5):
        side=160; cap=24; sample=pairs[k*10:(k+1)*10]
        sheet=Image.new('RGB',(side*2,(side+cap)*len(sample)),'white'); d=ImageDraw.Draw(sheet)
        for i,(fake,real,error) in enumerate(sample):
            y=i*(side+cap)
            for j,p in enumerate((fake,real)):
                with Image.open(p) as img: sheet.paste(ImageOps.fit(img.convert('RGB'),(side,side)),(j*side,y))
            d.text((3,y+side+3),fake.stem,fill='black'); d.text((side+3,y+side+3),real.stem[:20],fill='black')
        sheet.save(PREVIEW/f'generated_vs_real_{k+1:02d}.jpg',quality=92)
    print('왼쪽 생성, 오른쪽 실제: 픽셀 차이 기준 참고 비교이며 복사 여부를 증명하지 않습니다.',flush=True)
    display(Image.open(PREVIEW/'generated_masked_50.jpg').resize((900,510)))

    result=RUN_ROOT/'gan_masked_20261101_run01_results.zip'; tmp=result.with_suffix('.zip.part')
    if result.exists():
        with zipfile.ZipFile(result) as z:
            if z.testzip() is not None: raise RuntimeError('기존 결과 ZIP이 손상됐습니다.')
        print('기존 검증된 결과 ZIP 유지:',result,flush=True)
    else:
        with zipfile.ZipFile(tmp,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=4) as z:
            for root in (PREPARED,GAN,PREVIEW,LOGS):
                for path in root.rglob('*'):
                    if path.is_file() and path.suffix.lower() not in {'.pt','.pth','.tmp'}:
                        z.write(path,path.relative_to(RUN_ROOT).as_posix())
            z.write(lock,lock.relative_to(RUN_ROOT).as_posix())
        with zipfile.ZipFile(tmp) as z:
            bad=z.testzip()
            if bad: raise RuntimeError(f'ZIP 무결성 오류: {bad}')
        os.replace(tmp,result)
    status={'status':'completed','generated_count':50,'gpu':gpu,'result_zip':str(result),
            'warning':'마스크의 모양과 색이 함께 생성될 수 있으며 생성 품질은 별도 육안 검토 필요'}
    (RUN_ROOT/'run_status.json').write_text(json.dumps(status,ensure_ascii=False,indent=2),encoding='utf-8')
    print('완료! 결과 ZIP:',result,flush=True)
except BaseException as exc:
    status={'status':'failed','error_type':type(exc).__name__,'error':str(exc),'traceback':traceback.format_exc()}
    print('실험 오류:',status['traceback'],flush=True)
    if RUN_ROOT is not None:
        try:
            RUN_ROOT.mkdir(parents=True,exist_ok=True)
            (RUN_ROOT/'run_status.json').write_text(json.dumps(status,ensure_ascii=False,indent=2),encoding='utf-8')
        except Exception as err:
            print('Drive 오류 기록 실패:',err,flush=True)
finally:
    if MOUNTED:
        try:
            print('Drive 저장을 동기화합니다.',flush=True)
            drive.flush_and_unmount()
        except BaseException as err:
            print('Drive 동기화 확인 실패:',err,flush=True)
    print('G4 런타임 반납을 요청합니다.',flush=True)
    try: runtime.unassign()
    except BaseException as err:
        print('자동 반납 확인 실패. Colab에서 런타임 연결을 확인하세요:',err,flush=True)


## 결과

Drive의 `산공종설/results/gan_masked_20261101_run01/`에서 상태, 생성 사진, 접촉판과 결과 ZIP을 확인합니다.
